# Fine-tuning the twin model

Fine-tunes Qwen3-1.7B on a curated voice dataset with Unsloth QLoRA and exports a
q4_K_M GGUF that drops into the serving Space as a filename swap. The dataset teaches
tone, refusal behavior, and injection resistance — not facts. Grounding stays with
retrieval at runtime.

Runs on a free Colab T4 (Runtime -> Change runtime type -> T4 GPU). Kaggle works as a
fallback. Nothing here contains personal data: the curated dataset is uploaded at
runtime and never stored in this notebook.

## 1. Install

Unsloth pulls in the training stack (PEFT, TRL, bitsandbytes).

In [ ]:
%pip install -q unsloth

## 2. Load the base model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

## 3. Attach QLoRA adapters

Small rank for a small dataset; the base weights stay frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Upload the curated dataset

Run the harness locally (`npm run twin:dataset`), curate the rows you keep into
`curated.jsonl`, then upload that file here. Each line is one chat sample:
`{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}`.
Hold out ~10% for eval so overfitting shows up.

In [ ]:
from google.colab import files

files.upload()  # select curated.jsonl

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files="curated.jsonl", split="train")
split = data.train_test_split(test_size=0.1, seed=3407)
train_data, eval_data = split["train"], split["test"]
print(len(train_data), "train,", len(eval_data), "eval")

## 5. Format with the Qwen3 chat template (thinking off)

The system message already carries the retrieved CONTEXT block, exactly as the runtime
builds it, so the model learns to answer from context and decline without it.

In [ ]:
def format_sample(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_data = train_data.map(format_sample)
eval_data = eval_data.map(format_sample)

## 6. Train

Two epochs to start. Watch the eval loss between epochs — if it turns up while train
loss keeps falling, stop at the lower one. Small datasets overfit fast.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)
trainer.train()

## 7. Eyeball a few held-out prompts

Read the tone, not just the loss. Does it sound like the persona, and does it decline
when the context is thin?

In [ ]:
FastLanguageModel.for_inference(model)

def generate(messages):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    ).to("cuda")
    output = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7)
    return tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)

for row in eval_data.select(range(min(3, len(eval_data)))):
    prompt = row["messages"][:-1]
    print("Q:", prompt[-1]["content"])
    print("Tuned:", generate(prompt))
    print("-" * 40)

## 8. Export a q4_K_M GGUF

Merges the adapters into the base and quantizes to the same format the Space serves.

In [ ]:
model.save_pretrained_gguf(
    "twin-model-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

## 8b. Export the raw LoRA adapter

Adapter-serving hosts (Cloudflare Workers AI's free tier, for one) take this instead of a
merged GGUF: upload the adapter, then name its base model per request. One catch — such
hosts only serve adapters trained against a base in their own catalog, so check the host's
supported bases first and, if needed, change the base in step 2 before training.


In [ ]:
model.save_pretrained("twin-lora")
tokenizer.save_pretrained("twin-lora")

import shutil

shutil.make_archive("twin-lora", "zip", "twin-lora")
files.download("twin-lora.zip")


## 9. Upload to Hugging Face

The token comes from Colab secrets (key icon in the sidebar, name `HF_TOKEN`) and is
never printed.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
api.upload_folder(
    folder_path="twin-model-gguf",
    repo_id="percy80/twin-llm-model",
    repo_type="model",
)

## 10. Serve it

Two interchangeable paths. The site's client is host-agnostic — it needs an
OpenAI-compatible URL (`TWIN_MODEL_URL`), a bearer key (`TWIN_MODEL_API_KEY`), and, for
gateway hosts only, a model name (`TWIN_MODEL_NAME`):

1. **Self-hosted llama.cpp** (`services/twin-model/`): point the Dockerfile's `ADD` line at
   the new GGUF on `percy80/twin-llm-model`, deploy the container wherever it can run, then
   verify `GET /health` and one authed `POST /v1/chat/completions`.
2. **Adapter host (e.g. Cloudflare Workers AI)**: upload the `twin-lora` adapter as a
   fine-tune, set `TWIN_MODEL_NAME` to the fine-tune's model id, and point `TWIN_MODEL_URL`
   at the account's OpenAI-compatible endpoint.

Nothing else in the portfolio repo changes.
